# 9주차 워크북 3 — 촬영 이미지 특성 확인 · 96×96 리사이즈 · 이미지 갤러리 앱

**과목** 인공지능융합활용 | **실습 환경** Google Colab (설치할 것 없음, 브라우저만 있으면 됨)

ESP32-S3 카메라로 찍어 microSD에 저장한 사진을 **AI 학습용 데이터**로 바꾸는 과정을 따라 합니다.

| 단계 | 하는 일 | 왜 필요한가 |
|---|---|---|
| 0 | 사진 불러오기 (SD 사진 업로드 또는 샘플) | Colab 컴퓨터로 파일을 옮긴다 |
| 1 | 이미지 특성 확인 (크기·해상도·밝기·선명도) | 학습에 쓸 수 없는 사진(어둡거나 흐림)을 골라낸다 |
| 2 | 96×96 리사이즈 | AI 모델은 **모든 입력 크기가 같아야** 한다 |
| 3 | 이미지 갤러리 앱 만들기 | 원본과 변환 결과를 한눈에 비교·확인한다 |
| 4 | 라벨 폴더로 정리·내려받기 | 다음 주 모델 학습 준비 |

**사용 방법**
1. 메뉴 **파일 → Drive에 사본 저장** (내 사본에서 작업해야 저장됩니다)
2. 위에서부터 셀을 하나씩 선택하고 **Shift + Enter** 로 실행
3. `____` (밑줄 4개)로 표시된 **빈칸**을 채운 뒤 실행 — 막히면 완성본 노트북을 참고

## 0. 사진 불러오기

`PHOTO_SOURCE` 값을 골라 실행합니다.

- `"upload"` : 내 PC에 복사해 둔 SD 카드 사진(IMG_0001.jpg …)을 **여러 장 선택** 또는 **zip 1개**로 업로드
- `"sample"` : 카메라 실습을 못 했거나 연습할 때 — 강의 샘플 사진 6장 사용 (320×240, 카메라 QVGA와 같은 크기)
- `"drive"`  : 구글 드라이브의 폴더에서 읽기 (`DRIVE_FOLDER` 경로 입력)

> SD 카드 사진을 PC로 옮기는 법: 보드 전원을 끄고 microSD를 빼서 카드 리더기로 PC에 연결 → `photos` 폴더의 jpg 복사

In [ ]:
# ▶ 0-1. 사진 가져오기 ---------------------------------------------
PHOTO_SOURCE = "sample"      # "upload" / "sample" / "drive" 중 하나
DRIVE_FOLDER = "/content/drive/MyDrive/week09_photos"   # "drive" 선택 시 사용

import os, io, glob, shutil, zipfile, urllib.request
PHOTO_DIR = "/content/photos" if os.path.exists("/content") else "./photos"
os.makedirs(PHOTO_DIR, exist_ok=True)

def extract_zip(data):
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        for n in z.namelist():
            if n.lower().endswith((".jpg", ".jpeg", ".png")) and not n.startswith("__MACOSX"):
                with open(os.path.join(PHOTO_DIR, os.path.basename(n)), "wb") as f:
                    f.write(z.read(n))

def make_samples_offline():
    # 인터넷이 안 될 때: Colab에 기본 설치된 scikit-image 예제 사진으로 샘플 생성
    from skimage import data
    from PIL import Image, ImageEnhance, ImageFilter
    def qvga(a):
        im = Image.fromarray(a).convert("RGB"); w, h = im.size
        tw = min(w, int(h * 4 / 3)); th = int(tw * 3 / 4)
        return im.crop(((w - tw)//2, (h - th)//2, (w - tw)//2 + tw, (h - th)//2 + th)).resize((320, 240))
    ims = [qvga(data.coffee()), qvga(data.chelsea()), qvga(data.rocket())]
    ims += [ImageEnhance.Brightness(ims[0]).enhance(0.35), Image.blend(ims[0], Image.new("RGB", ims[0].size, "white"), 0.7),
            ims[1].filter(ImageFilter.GaussianBlur(3))]
    for i, im in enumerate(ims, 1):
        im.save(f"{PHOTO_DIR}/IMG_{i:04d}.jpg", quality=85)

if PHOTO_SOURCE == "upload":
    from google.colab import files
    up = files.upload()                       # 파일 선택 창이 열림 (여러 장 또는 zip)
    for name, data in up.items():
        if name.lower().endswith(".zip"): extract_zip(data)
        else: open(os.path.join(PHOTO_DIR, name), "wb").write(data)
elif PHOTO_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    for p in glob.glob(DRIVE_FOLDER + "/*"):
        if p.lower().endswith((".jpg", ".jpeg", ".png")): shutil.copy(p, PHOTO_DIR)
else:
    try:
        extract_zip(urllib.request.urlopen("https://raw.githubusercontent.com/itlects/2026-1_ai-convergence-apply/main/week09_esp32_ai_image/colab/sample_images.zip", timeout=20).read())
        print("GitHub 샘플 사진 다운로드 완료")
    except Exception as e:
        print("다운로드 실패 → 오프라인 샘플 생성:", e)
        make_samples_offline()

photo_files = sorted(glob.glob(PHOTO_DIR + "/*.jpg") + glob.glob(PHOTO_DIR + "/*.jpeg") + glob.glob(PHOTO_DIR + "/*.png"))
print(f"사진 {len(photo_files)}장:", [os.path.basename(p) for p in photo_files])

## 1. 촬영 이미지 특성 확인

### 1-1. 사진 한 장 열어 보기
디지털 이미지는 **숫자(픽셀) 표**입니다. 컬러 사진의 픽셀 하나는 빨강(R)·초록(G)·파랑(B) 세 숫자(0~255)로 되어 있습니다.

| 용어 | 뜻 | 예 (QVGA 사진) |
|---|---|---|
| 해상도 | 가로 × 세로 픽셀 수 | 320 × 240 |
| 채널 | 픽셀 하나의 숫자 개수 | RGB = 3 |
| 배열 모양(shape) | (세로, 가로, 채널) | (240, 320, 3) |
| 데이터 형(dtype) | 숫자 종류 | uint8 (0~255 정수) |
| 파일 크기 | JPEG로 **압축된** 저장 크기 | 약 8~20 KB |

In [ ]:
# ▶ 1-1. 첫 번째 사진의 기본 정보
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

path = photo_files[0]
img = Image.open(path)
arr = np.array(img)                       # 이미지 → 숫자 배열

print("파일 이름 :", os.path.basename(path))
print("파일 형식 :", img.format)           # JPEG
print("해상도    :", img.size, "(가로, 세로)")
print("색상 모드 :", img.mode)             # RGB
print("배열 모양 :", arr.____)             # (세로, 가로, 채널)   ← 빈칸: 배열의 모양 속성
print("데이터 형 :", arr.dtype)
print("파일 크기 :", os.path.getsize(path), "바이트")
print("메모리상 크기:", arr.size, "바이트 (압축 전)")
print("압축률    : 약", round(arr.size / os.path.getsize(path), 1), "배")

plt.imshow(img); plt.title(os.path.basename(path)); plt.axis("off"); plt.show()

### 1-2. 픽셀 값과 RGB 채널 들여다보기

In [ ]:
# ▶ 1-2. 왼쪽 위 3×3 픽셀의 실제 숫자 + R/G/B 채널 분리
print("왼쪽 위 3×3 픽셀 [R, G, B]:\n", arr[:3, :3])

fig, ax = plt.subplots(1, 4, figsize=(16, 3.5))
ax[0].imshow(arr); ax[0].set_title("RGB 원본")
for i, (name, cmap) in enumerate([("R (빨강)", "Reds"), ("G (초록)", "Greens"), ("B (파랑)", "Blues")]):
    ax[i + 1].imshow(arr[:, :, i], cmap=cmap); ax[i + 1].set_title(name)
for a in ax: a.axis("off")
plt.show()

### 1-3. 밝기 히스토그램
히스토그램은 **0(검정)~255(흰색) 밝기 값이 각각 몇 개 있는지** 세어 그린 그래프입니다.
- 왼쪽에 몰려 있음 → 어두운 사진, 오른쪽에 몰려 있음 → 너무 밝은 사진

In [ ]:
# ▶ 1-3. 사진 3장의 밝기 히스토그램 비교
import matplotlib
try:
    import matplotlib.font_manager as fm   # 한글 글꼴 (Colab)
    if not [f for f in fm.findSystemFonts() if "Nanum" in f]:
        os.system("apt-get -qq -y install fonts-nanum > /dev/null 2>&1")
    for f in fm.findSystemFonts():
        if "NanumGothic.ttf" in f: fm.fontManager.addfont(f); plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception as e:
    print("한글 글꼴 설정 생략:", e)

fig, ax = plt.subplots(2, 3, figsize=(15, 7))
for k, p in enumerate(photo_files[:6:2] if len(photo_files) >= 5 else photo_files[:3]):
    g = np.array(Image.open(p).convert("L"))          # 흑백(밝기)으로 변환
    ax[0, k].imshow(g, cmap="gray", vmin=0, vmax=255); ax[0, k].set_title(os.path.basename(p)); ax[0, k].axis("off")
    ax[1, k].hist(g.ravel(), bins=32, range=(0, 255), color="gray")
    ax[1, k].set_title(f"평균 밝기 {g.mean():.0f}"); ax[1, k].set_xlim(0, 255)
plt.tight_layout(); plt.show()

### 1-4. 모든 사진의 특성표 만들기 + 불량 사진 찾기
- **평균 밝기**: 60 미만이면 '어두움', 200 초과면 '너무 밝음'
- **선명도**: 이웃 픽셀 차이(라플라시안)의 분산. 값이 작을수록 흐림(초점 안 맞음·흔들림)

In [ ]:
# ▶ 1-4. 특성표
import pandas as pd

def sharpness(gray):
    g = gray.astype(float)
    lap = -4 * g[1:-1, 1:-1] + g[:-2, 1:-1] + g[2:, 1:-1] + g[1:-1, :-2] + g[1:-1, 2:]
    return lap.var()

DARK, BRIGHT, BLUR = 60, 200, 100      # 기준값 (사진을 보며 바꿔 보세요)
rows = []
for p in photo_files:
    im = Image.open(p).convert("RGB"); a = np.array(im); g = np.array(im.convert("L"))
    bright, sharp = g.mean(), sharpness(g)
    problems = []
    if bright < DARK:   problems.append("어두움")
    if bright > ____:   problems.append("너무 밝음")       # ← 빈칸: 밝음 기준 변수
    if sharp < BLUR:    problems.append("흐림")
    rows.append(dict(파일=os.path.basename(p), 가로=im.width, 세로=im.height,
                     파일KB=round(os.path.getsize(p) / 1024, 1),
                     평균R=round(a[..., 0].mean()), 평균G=round(a[..., 1].mean()), 평균B=round(a[..., 2].mean()),
                     평균밝기=round(bright), 선명도=round(sharp), 판정=", ".join(problems) or "OK"))
df = pd.DataFrame(rows)
df

> **생각해 보기**: 파일 크기(KB)와 선명도는 어떤 관계가 있나요? (JPEG는 무늬가 복잡할수록 압축이 덜 되어 파일이 커집니다.)

## 2. 96×96 리사이즈

AI 이미지 분류 모델(다음 주 Edge Impulse 등)은 **입력 크기가 고정**되어 있습니다. 작은 마이크로컨트롤러에서 돌리기 위해 보통 **96×96** 처럼 작게 씁니다.

| | 원본 QVGA | 96×96 컬러 | 96×96 흑백 |
|---|---|---|---|
| 픽셀 수 | 76,800 | 9,216 | 9,216 |
| 숫자 개수 (모델 입력) | 230,400 | 27,648 | 9,216 |

**그냥 줄이면** 4:3 사진이 1:1로 찌그러집니다 → 먼저 **가운데를 정사각형으로 자르고(center crop)** 줄입니다.

![자르기 vs 찌그러뜨리기](https://raw.githubusercontent.com/itlects/2026-1_ai-convergence-apply/main/week09_esp32_ai_image/diagrams/14_crop_vs_stretch.png)

In [ ]:
# ▶ 2-1. 중앙 자르기 + 리사이즈 함수
def center_crop_resize(im, size=96, gray=False):
    im = im.convert("RGB")
    w, h = im.size
    s = min(w, h)                              # 정사각형 한 변 = 짧은 쪽 길이
    left, top = (w - s) // 2, (h - s) // 2     # 가운데 정렬
    im = im.crop((left, top, left + s, top + s))
    im = im.resize((____, ____), Image.BILINEAR)   # ← 빈칸: 목표 크기 (변수 이름 사용)
    if gray:
        im = im.convert("L")
    return im

src = Image.open(photo_files[0])
stretch = src.resize((96, 96))
crop96  = center_crop_resize(src)
gray96  = center_crop_resize(src, gray=True)

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, (im, t) in zip(ax, [(src, f"원본 {src.size[0]}×{src.size[1]}"), (stretch, "그냥 줄이기(찌그러짐)"),
                           (crop96, "중앙 자르기 → 96×96"), (gray96, "흑백 96×96")]):
    a.imshow(im, cmap="gray" if im.mode == "L" else None, interpolation="nearest"); a.set_title(t); a.axis("off")
plt.show()
print("96x96 컬러 배열:", np.array(crop96).shape, " / 흑백 배열:", np.array(gray96).shape)

In [ ]:
# ▶ 2-2. 모든 사진 변환 → PNG로 저장 + 크기 비교
OUT96 = PHOTO_DIR + "_96"
os.makedirs(OUT96, exist_ok=True)
GRAY = False                                   # True로 바꾸면 흑백으로 저장

rows = []
for p in photo_files:
    im96 = center_crop_resize(Image.open(p), 96, GRAY)
    out = os.path.join(OUT96, os.path.splitext(os.path.basename(p))[0] + "_96.png")
    im96.save(out)
    rows.append(dict(원본=os.path.basename(p), 원본KB=round(os.path.getsize(p) / 1024, 1),
                     변환=os.path.basename(out), 변환KB=round(os.path.getsize(out) / 1024, 1),
                     변환해상도=f"{im96.width}x{im96.height}", 모드=im96.mode))
pd.DataFrame(rows)

In [ ]:
# ▶ 2-3. 모델 입력 형태로 바꾸기 (0~255 정수 → 0~1 실수)
x = np.array(Image.open(glob.glob(OUT96 + "/*.png")[0])).astype("float32") / 255.0
print("모델 입력 한 장의 모양:", x.shape, " 최솟값", x.min(), " 최댓값", round(float(x.max()), 3))
batch = np.stack([np.array(Image.open(p)).astype("float32") / 255.0 for p in sorted(glob.glob(OUT96 + "/*.png"))])
print("전체 데이터 묶음(batch) 모양:", batch.shape, "→ (사진 수, 세로, 가로, 채널)")

## 3. 이미지 갤러리 앱 만들기

### 3-1. 간단 갤러리 (그림 격자)

In [ ]:
# ▶ 3-1. 원본 / 96×96 을 위아래로 보여 주는 갤러리 함수
def show_gallery(files, cols=6, size96=False):
    n = len(files); rows_ = (n + cols - 1) // cols
    fig, ax = plt.subplots(rows_, cols, figsize=(cols * 2.6, rows_ * 2.6), squeeze=False)
    for i, a in enumerate(ax.ravel()):
        a.axis("off")
        if i < n:
            im = Image.open(files[i])
            a.imshow(im, cmap="gray" if im.mode == "L" else None, interpolation="nearest" if size96 else None)
            a.set_title(f"{os.path.basename(files[i])}\n{im.width}×{im.height}", fontsize=9)
    plt.tight_layout(); plt.show()

show_gallery(photo_files)
show_gallery(sorted(glob.glob(OUT96 + "/*.png")), size96=True)

### 3-2. 웹 갤러리 앱 (Gradio)
**Gradio**는 파이썬 함수 몇 줄로 웹 앱 화면을 만들어 주는 라이브러리입니다. 실행하면 셀 아래에 앱이 나타나고, `https://xxxx.gradio.live` **공유 링크**가 생겨 스마트폰에서도 열 수 있습니다 (72시간 유지).

앱 기능: 사진 여러 장 업로드 → 원본 갤러리 · 96×96 갤러리 · 특성표 표시 → 변환 결과 zip 내려받기

> 설치에 30초~1분 걸립니다. 앱을 멈추려면 셀의 ■(중지) 버튼을 누릅니다.

In [ ]:
# ▶ 3-2-1. Gradio 설치
!pip -q install gradio

In [ ]:
# ▶ 3-2-2. 갤러리 앱 정의 & 실행
import gradio as gr, tempfile, time

def analyze(paths, size, gray, dark, blur):
    paths = paths or []
    originals, converted, rows = [], [], []
    tmp = tempfile.mkdtemp()
    for p in paths:
        p = p if isinstance(p, str) else p.name
        im = Image.open(p).convert("RGB"); g = np.array(im.convert("L"))
        out_im = center_crop_resize(im, int(size), gray)
        name = os.path.splitext(os.path.basename(p))[0]
        out_path = os.path.join(tmp, f"{name}_{int(size)}.png"); out_im.save(out_path)
        b, s = g.mean(), sharpness(g)
        flag = "어두움" if b < dark else ("너무 밝음" if b > 200 else ("흐림" if s < blur else "OK"))
        originals.append((p, f"{os.path.basename(p)} ({im.width}×{im.height})"))
        converted.append((out_path, f"{name} → {int(size)}×{int(size)}"))
        rows.append([os.path.basename(p), f"{im.width}×{im.height}", round(os.path.getsize(p) / 1024, 1),
                     round(b), round(s), flag])
    zip_path = None
    if converted:
        zip_path = os.path.join(tmp, f"converted_{int(time.time())}.zip")
        with zipfile.ZipFile(zip_path, "w") as z:
            for fp, _ in converted: z.write(fp, os.path.basename(fp))
    table = pd.DataFrame(rows, columns=["파일", "해상도", "KB", "평균밝기", "선명도", "판정"])
    return originals, converted, table, zip_path

with gr.Blocks(title="9주차 이미지 갤러리") as app:
    gr.Markdown("## 📷 ESP32-S3 촬영 이미지 갤러리 — 특성 확인 · 96×96 변환")
    with gr.Row():
        files_in = gr.File(label="사진 업로드 (여러 장 선택)", file_count="multiple", file_types=["image"])
        with gr.Column():
            size_in = gr.Slider(32, 160, value=96, step=16, label="변환 크기 (정사각형)")
            gray_in = gr.Checkbox(label="흑백으로 변환", value=False)
            dark_in = gr.Slider(0, 120, value=60, step=5, label="'어두움' 기준 밝기")
            blur_in = gr.Slider(0, 500, value=100, step=10, label="'흐림' 기준 선명도")
            run = gr.Button("갤러리 만들기", variant="primary")
    gal_orig = gr.Gallery(label="원본", columns=4, height="auto")
    gal_96 = gr.Gallery(label="변환 결과", columns=6, height="auto")
    table_out = gr.Dataframe(label="특성표")
    zip_out = gr.File(label="변환 이미지 zip 내려받기")
    run.click(analyze, [files_in, size_in, gray_in, dark_in, blur_in], [gal_orig, gal_96, table_out, zip_out])

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
if IN_COLAB:
    app.launch(share=True, debug=False)
else:
    # (Colab이 아닌 곳에서 시험할 때) 앱 함수만 샘플로 확인
    o, c, t, z = analyze(photo_files, 96, False, 60, 100)
    print(len(o), "장 처리 완료, zip:", os.path.basename(z)); display(t)

## 4. 다음 주 학습을 위한 데이터 정리

AI가 배우려면 사진마다 **정답 이름표(라벨)** 가 필요합니다. 가장 흔한 방법은 **라벨 이름의 폴더**에 사진을 나눠 넣는 것입니다.

```
dataset/
 ├─ class_A/   IMG_0001_96.png, IMG_0004_96.png ...
 └─ class_B/   IMG_0002_96.png ...
```

아래 `LABELS` 에 우리 팀 프로젝트의 클래스 이름과 사진을 적어 보세요. 판정이 'OK'가 아닌 사진은 빼는 것이 좋습니다.

**데이터 기획 체크** — 클래스는 2~4개로 시작 · 클래스마다 사진 수를 비슷하게(최소 30장 이상 목표) · 배경·조명·각도를 다양하게

In [ ]:
# ▶ 4-1. 라벨 폴더로 정리하고 zip 내려받기
LABELS = {                                       # 클래스 이름: [원본 파일 이름 목록]
    "coffee": ["IMG_0001.jpg", "IMG_0004.jpg", "IMG_0005.jpg"],
    "cat":    ["IMG_0002.jpg", "IMG_0006.jpg"],
    "rocket": ["IMG_0003.jpg"],
}
SKIP_BAD = True                                   # 판정이 OK가 아닌 사진 제외

DATASET = "/content/dataset" if os.path.exists("/content") else "./dataset"
shutil.rmtree(DATASET, ignore_errors=True)
status = dict(zip(df["파일"], df["판정"]))
for label, names in LABELS.items():
    os.makedirs(os.path.join(DATASET, label), exist_ok=True)
    for n in names:
        src = os.path.join(OUT96, os.path.splitext(n)[0] + "_96.png")
        if not os.path.exists(src): print("없음:", n); continue
        if SKIP_BAD and status.get(n, "OK") != "OK": print(f"제외({status[n]}):", n); continue
        shutil.copy(src, os.path.join(DATASET, label))

for label in LABELS:
    print(f"{label:8s}: {len(os.listdir(os.path.join(DATASET, label)))}장")

shutil.make_archive(DATASET, "zip", DATASET)
try:
    from google.colab import files; files.download(DATASET + ".zip")
except ImportError:
    print("zip 생성:", DATASET + ".zip")

## 5. 정리 질문 (워크북 제출)

1. 우리 카메라 사진의 해상도·파일 크기·배열 모양(shape)은? 메모리상 크기는 파일 크기의 몇 배였나요? 그 이유는?
2. 특성표에서 '어두움'·'흐림'으로 판정된 사진은 몇 장이었나요? 촬영할 때 어떻게 하면 줄일 수 있을까요?
3. 320×240 사진을 그냥 96×96으로 줄이면 어떤 문제가 생기나요? 중앙 자르기의 단점은?
4. 96×96 흑백 이미지 한 장이 모델에 들어가는 숫자는 몇 개인가요? 컬러라면?
5. 우리 팀 융합 프로젝트에서 카메라로 구분하고 싶은 클래스 2~4개와, 클래스당 촬영 계획(장수·장소·조명)을 적어 보세요.